# canonical_data — one-shot seeder

Ported from `caspers-kitchens/stages/canonical_data.ipynb` at commit 8c756ac on 2026-04-20.
Caspers is retired — this is now the authoritative copy.

**Invoked by**: `setup-lakebase` job, `canonical_data` task (after `bootstrap_datagen`).


### Canonical Data

This notebook bootstraps the Domino's Digital Twin canonical data into the provided catalog and schema.

Unlike the old generator, this uses pre-generated data from the canonical dataset that is replayed at configurable speeds.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
EVENTS_VOLUME = dbutils.widgets.get("EVENTS_VOLUME")
SIMULATOR_SCHEMA = dbutils.widgets.get("SIMULATOR_SCHEMA")
START_DAY = dbutils.widgets.get("START_DAY") if dbutils.widgets.get("START_DAY") else "20"
SPEED_MULTIPLIER = dbutils.widgets.get("SPEED_MULTIPLIER") if dbutils.widgets.get("SPEED_MULTIPLIER") else "1.0"
SCHEDULE_MINUTES = dbutils.widgets.get("SCHEDULE_MINUTES") if dbutils.widgets.get("SCHEDULE_MINUTES") else "5"

##### Create simulator schema and volumes (catalog must already exist)

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS ${CATALOG}.${SIMULATOR_SCHEMA};
CREATE VOLUME IF NOT EXISTS ${CATALOG}.${SIMULATOR_SCHEMA}.${EVENTS_VOLUME};
CREATE VOLUME IF NOT EXISTS ${CATALOG}.${SIMULATOR_SCHEMA}.misc;

##### Create tables from canonical dataset parquet files

Load dimensional data from the canonical dataset (not from ./data/dimensional)

In [ ]:
import pandas as pd

# Twins Phase 1: read from bootstrap_datagen output (UC Volume).
# Caspers original path was ../data/canonical/canonical_dataset/; twins
# pre-stages the parquets in the canonical_seed Volume via bootstrap_datagen.
SEED_DIR = f"/Volumes/{CATALOG}/{SIMULATOR_SCHEMA}/canonical_seed"

# Load dimension tables from canonical dataset
spark.createDataFrame(pd.read_parquet(f"{SEED_DIR}/brands.parquet")) \
    .write.mode("overwrite").saveAsTable(f"{CATALOG}.{SIMULATOR_SCHEMA}.brands")

spark.createDataFrame(pd.read_parquet(f"{SEED_DIR}/locations.parquet")) \
    .write.mode("overwrite").saveAsTable(f"{CATALOG}.{SIMULATOR_SCHEMA}.locations")

spark.createDataFrame(pd.read_parquet(f"{SEED_DIR}/menus.parquet")) \
    .write.mode("overwrite").saveAsTable(f"{CATALOG}.{SIMULATOR_SCHEMA}.menus")

spark.createDataFrame(pd.read_parquet(f"{SEED_DIR}/categories.parquet")) \
    .write.mode("overwrite").saveAsTable(f"{CATALOG}.{SIMULATOR_SCHEMA}.categories")

spark.createDataFrame(pd.read_parquet(f"{SEED_DIR}/items.parquet")) \
    .write.mode("overwrite").saveAsTable(f"{CATALOG}.{SIMULATOR_SCHEMA}.items")

spark.createDataFrame(pd.read_parquet(f"{SEED_DIR}/brand_locations.parquet")) \
    .write.mode("overwrite").saveAsTable(f"{CATALOG}.{SIMULATOR_SCHEMA}.brand_locations")

print("✅ Dimensional tables created from canonical_seed volume (Phase 1: SF-only, 22 locations)")

##### Start canonical data replay

Create a scheduled job that runs the canonical generator notebook at the specified interval

In [ ]:
# Twins Phase 1: DAB declares `twins-datagen-replay` as the scheduled replay
# job (PAUSED at deploy). Kick off a one-time run now so the events volume
# is populated before the DLT pipeline starts consuming.
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
jobs = [j for j in w.jobs.list(name="twins-datagen-replay")]
if jobs:
    job_id = jobs[0].job_id
    run = w.jobs.run_now(job_id=job_id)
    print(f"🚀 Triggered one-time run of twins-datagen-replay job_id={job_id} run_id={run.run_id}")
else:
    print("⚠️  twins-datagen-replay job not found — skipping one-shot trigger.")
    print("   DLT pipeline will wait for data until the replay is unpaused manually.")

##### Blocking cell to wait for some data to arrive at the volume.

The lakeflow declarative pipeline that comes next infers the schema from existing data.

Lakeflow Jobs doesn't have a file arrival trigger at the task level (yet?)

In [ ]:
import time

# Construct the path to the volume where JSONs will arrive
volume_path = f"/Volumes/{CATALOG}/{SIMULATOR_SCHEMA}/{EVENTS_VOLUME}"

def wait_for_data(path, timeout=300, poll_interval=5):
    """
    Wait until at least one file appears in the given path.
    Args:
        path (str): The directory to watch.
        timeout (int): Maximum seconds to wait.
        poll_interval (int): Seconds between checks.
    Raises:
        TimeoutError: If no file appears within the timeout.
    """
    start = time.time()
    while time.time() - start < timeout:
        files = dbutils.fs.ls(path)
        if any(f.size > 0 for f in files if not f.path.endswith('/')):
            print("✅ Data arrived. Safe to proceed.")
            return
        time.sleep(poll_interval)
    raise TimeoutError(f"No data found in {path} after {timeout} seconds.")

wait_for_data(volume_path)